In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve, roc_curve, auc
from imblearn.over_sampling import SMOTE #불균형 데이터 처리
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LayerNormalization, Dropout, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.callbacks import EarlyStopping

In [3]:
# Positional Encoding Layer
class PositionalEncoding(tf.keras.layers.Layer): #Transformer가 이 데이터가 7일 중 몇 번째 날의 것인지를 이해할 수 있도록 위치 정보 추가
    def call(self, x): # x의 형태: (batch_size, 7, 9)
        seq_len = tf.shape(x)[1] # 시퀀스 길이 (7일)
        d_model = tf.shape(x)[2] # 특성 개수 (9개)
        pos = tf.range(seq_len, dtype=tf.float32)[:, tf.newaxis] #pos: 각 시간 단계 (0일차, 1일차, ..., 6일차)
        i = tf.range(d_model, dtype=tf.float32)[tf.newaxis, :] #i: 각 특성 차원 (0번째 특성, 1번째 특성, ..., 8번째 특성)
        angle_rates = 1 / tf.pow(10000., (2 * (i // 2)) / tf.cast(d_model, tf.float32)) #각도 계산률
        angle_rads = pos * angle_rates #각도 계산
        sines = tf.sin(angle_rads[:, 0::2]) # 짝수 인덱스에 sin 적용
        cosines = tf.cos(angle_rads[:, 1::2])  # 짝수 인덱스에 sin 적용
        pos_encoding = tf.concat([sines, cosines], axis=-1) #Sin/Cos 결합
        pos_encoding = pos_encoding[tf.newaxis, ...] # (1, 7, 9)로 변환
        return x + pos_encoding

In [4]:
# Transformer Block
def transformer_block(inputs, num_heads, key_dim, ff_dim, dropout_rate=0.1):
    attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(inputs, inputs) #7일간의 데이터에서 중요한 날들 간의 관계를 학습한 결과
    out1 = LayerNormalization(epsilon=1e-6)(inputs + attn_output) # 원본 데이터 + Attention 결과를 더함. 원본 정보 손실 방지
    #비선형 변환 수행
    ffn = tf.keras.Sequential([ #Sequential 모델 시작: 레이어들을 순차적으로 연결하는 모델 생성 시작
        Dense(ff_dim, activation='relu'), #완전연결층(9차원 -> 64개 뉴런(ff_dim)의 신경망 생성)
        Dense(inputs.shape[-1])  #완전연결층(64차원 -> 9개 뉴런으로 복원)
    ]) #Sequential모델 종료
    ffn_output = ffn(out1) #비선형 변환된 결과 (복잡한 패턴이 학습됨)
    out2 = LayerNormalization(epsilon=1e-6)(out1 + ffn_output) #두 번째 정규화된 출력
    return Dropout(dropout_rate)(out2)
#입력 (7일×9특성) 
#→ Attention (날짜간 관계학습) 
#→ +원본 & 정규화 
#→ FFN (9→64→9, 복잡한패턴학습) 
#→ +이전결과 & 정규화 
#→ Dropout (과적합방지) 
#→ 출력


In [5]:
# Transformer Model
def build_transformer_model(input_shape, num_blocks=2): #Transformer 모델 생성
    inputs = Input(shape=input_shape) #Input 생성:(7일×9특성)
    x = PositionalEncoding()(inputs) #위치인코딩: (배치, 7, 9) 날짜정보 추가
    for _ in range(num_blocks): #num_blocks(2) 번 반복
        x = transformer_block(x, num_heads=2, key_dim=32, ff_dim=64) #Transformer 블록 적용: 앞서 정의한 transformer_block 함수 호출
    x = GlobalAveragePooling1D()(x) #7일간의 패턴을 하나의 대표값으로 요약
    x = Dropout(0.2)(x)
    x = Dense(32, activation='relu')(x)
    outputs = Dense(1, activation='sigmoid')(x)
    return Model(inputs, outputs)

In [6]:
def create_sequences(X, y, window_size=7):
    X_seq, y_seq = [], []
    for i in range(len(X) - window_size + 1):
        X_seq.append(X[i:i+window_size])
        y_seq.append(y[i+window_size-1])
    return np.array(X_seq), np.array(y_seq)
#시계열 데이터 변환: 일별 데이터를 7일 시퀀스로 변환
#예시 : 원본: [Day1, Day2, Day3, Day4, Day5, Day6, Day7, Day8, ...]
#변환: 
#- 시퀀스1: [Day1~Day7] → Day7 홍수위험도
#- 시퀀스2: [Day2~Day8] → Day8 홍수위험도

In [7]:
def train_transformer(csv_path="data/asos_seoul_daily_enriched.csv", model_path="models/transformer_flood_model.h5"):
    df = pd.read_csv(csv_path)
    features = ['avgTa', 'minTa', 'maxTa', 'sumRn', 'avgWs', 'avgRhm', 'avgTs', 'avgTd', 'avgPs'] #9개 기상 특성만 사용(Transformer가 시간 패턴을 스스로 학습 가능)
    target = 'flood_risk' #침수 위험 여부(0, 1)

    df = df.dropna(subset=features + [target]) #결측값 제거 후 numpy 배열로 변환
    X_raw = df[features].values
    y = df[target].values

    X_seq, y_seq = create_sequences(X_raw, y, window_size=7) # 일별 데이터를 7일 시퀀스로 변환

    X_train, X_test, y_train, y_test = train_test_split( #훈련용:테스트용=80:20
        X_seq, y_seq, test_size=0.2, random_state=42, stratify=y_seq
    )

    X_train_flat = X_train.reshape(X_train.shape[0], -1) # 2D로 평면화
    smote = SMOTE(random_state=42) #SMOTE를 이용한 불균형 데이터 처리. 침수 발생 사례가 적어서 인공 데이터 생성
    X_train_res, y_train_res = smote.fit_resample(X_train_flat, y_train)
    X_train_res = X_train_res.reshape(-1, 7, len(features))# 다시 3D로 복원

    model = build_transformer_model((7, len(features))) #모델 생성
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])

    early_stop = EarlyStopping(patience=3, restore_best_weights=True)
    history = model.fit( #모델 학습 : 15회
        X_train_res, y_train_res,
        epochs=15, batch_size=32,
        validation_split=0.2,
        callbacks=[early_stop]
    )

    y_pred_proba = model.predict(X_test).ravel() #테스트 데이터에 대해 모델이 홍수 발생 확률을 예측 후 평면화(ravel())
    y_pred = (y_pred_proba >= 0.5).astype(int) #확률값이 0.5 이상이면 True(1), 미만이면 False(0)로 변환

    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred)) #혼동행렬을 계산하고 콘솔에 출력 (실제값 vs 예측값의 교차표)
    print("\nClassification Report:\n", classification_report(y_test, y_pred)) #정밀도, 재현율, F1-점수 등의 분류 성능 지표들을 표 형태로 출력함
    print("\nROC AUC Score:", roc_auc_score(y_test, y_pred_proba)) #print("\nROC AUC Score:", roc_auc_score(y_test, y_pred_proba))

    model.save(model_path) #모델 저장
    print(f"모델 저장 완료: {model_path}")

    # 첫 번째 시각화: 학습 곡선 
    plt.figure(figsize=(10, 4))
    #첫 번째 서브플롯:Accuracy
    plt.subplot(1, 2, 1) 
    plt.plot(history.history['accuracy'], label='Train Acc')
    plt.plot(history.history['val_accuracy'], label='Val Acc')
    plt.title("Accuracy")
    plt.xlabel("Epochs")
    plt.legend()
    #두번째 서브플롯:Loss
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title("Loss")
    plt.xlabel("Epochs")
    plt.legend()

    plt.tight_layout()
    plt.savefig("outputs/transformer_train_metrics.png") #학습곡선 이미지 저장
    plt.show()

    # 두 번째 시각화: 혼동행렬(Confusion Matrix)
    plt.figure(figsize=(5, 4))
    sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.savefig("outputs/transformer_confusion_matrix.png") #혼동행렬 이미지 저장
    plt.show()

    # 세 번째 시각화: ROC + PR Curve
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba) #ROC 곡선을 위한 거짓양성률(FPR)과 참양성률(TPR)을 계산 (세 번째 값은 사용하지 않으므로 _로 무시)
    precision, recall, _ = precision_recall_curve(y_test, y_pred_proba) #PR 곡선을 위한 정밀도와 재현율을 계산
    roc_auc = auc(fpr, tpr) #FPR과 TPR로부터 ROC 곡선 아래 면적을 계산

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(fpr, tpr, label=f"ROC AUC = {roc_auc:.4f}")
    plt.plot([0, 1], [0, 1], 'k--')
    plt.title("ROC Curve")
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(recall, precision, color='green')
    plt.title("Precision-Recall Curve")
    plt.xlabel("Recall")
    plt.ylabel("Precision")

    plt.tight_layout()
    plt.savefig("outputs/transformer_roc_pr.png") #ROC + PR 이미지 저장
    plt.show()
